In [43]:
import numpy as np
import pandas as pd
from weli.layers import Dense
from weli.losses import MSE
from weli.models import Sequential, load_model
from weli.optimizers import Adam
import matplotlib.pyplot as plt

In [44]:
df = pd.read_csv("../Data/dataset_clean.csv")

In [45]:
df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,n_items,...,n_sellers,first_category,n_payments,payment_value,max_installments,review_score,late_delivery,promised_days,purchase_month,purchase_dow
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,1.0,...,1.0,housewares,3.0,38.71,1.0,4.0,0,15,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,1.0,...,1.0,perfumery,1.0,141.46,1.0,4.0,0,19,7,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO,1.0,...,1.0,auto,1.0,179.12,3.0,5.0,0,26,8,2
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,RN,1.0,...,1.0,pet_shop,1.0,72.20,1.0,5.0,0,26,11,5
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,SP,1.0,...,1.0,stationery,1.0,28.62,1.0,5.0,0,12,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96465,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,SP,1.0,...,1.0,health_beauty,1.0,85.08,3.0,5.0,0,18,3,3
96466,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,SP,1.0,...,1.0,baby,1.0,195.00,3.0,4.0,0,23,2,1
96467,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,BA,1.0,...,1.0,home_appliances_2,1.0,271.01,5.0,5.0,0,30,8,6
96468,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,RJ,2.0,...,1.0,computers_accessories,1.0,441.16,4.0,2.0,0,37,1,0


In [48]:
df.shape

(96470, 23)

In [46]:
features = df[
    [
        "n_items", "total_price", "total_freight", "total_weight_g", "n_sellers",
        "first_category", "customer_state",
        "n_payments", "payment_value", "max_installments",
        "promised_days", "purchase_month", "purchase_dow",
    ]
    ]


In [51]:
x = features.drop(columns=["first_category", "customer_state"])
y = df["late_delivery"]

In [ ]:
# traductions = {
#     "order_id": "id_commande",
#     "customer_id": "id_client",
#     "order_status": "statut_commande",
#     "order_purchase_timestamp": "date_achat",
#     "order_approved_at": "date_approbation",
#     "order_delivered_carrier_date": "date_remise_transporteur",
#     "order_delivered_customer_date": "date_livraison_client",
#     "order_estimated_delivery_date": "date_livraison_estimee",
#     "customer_state": "etat_client",
#     "n_items": "nombre_articles",
#     "total_price": "prix_total",
#     "total_freight": "frais_livraison",
#     "total_weight_g": "poids_total_g",
#     "n_sellers": "nombre_vendeurs",
#     "first_category": "premiere_categorie",
#     "n_payments": "nombre_paiements",
#     "payment_value": "montant_paiement",
#     "max_installments": "nombre_max_echeances",
#     "review_score": "note_avis",
#     "late_delivery": "livraison_en_retard",
#     "promised_days": "jours_promis",
#     "purchase_month": "mois_achat",
#     "purchase_dow": "jour_semaine_achat"
# }

# def renommer_colonnes(df):
#     df = df.copy()
#     df.columns = df.columns.str.strip()
#     return df.rename(columns=traductions)

# df = renommer_colonnes(df)

# print(df.columns.tolist())

['id_commande', 'id_client', 'statut_commande', 'date_achat', 'date_approbation', 'date_remise_transporteur', 'date_livraison_client', 'date_livraison_estimee', 'etat_client', 'nombre_articles', 'prix_total', 'frais_livraison', 'poids_total_g', 'nombre_vendeurs', 'premiere_categorie', 'nombre_paiements', 'montant_paiement', 'nombre_max_echeances', 'note_avis', 'livraison_en_retard', 'jours_promis', 'mois_achat', 'jour_semaine_achat']


In [40]:
df["statut_commande"].value_counts()


statut_commande
delivered    96470
Name: count, dtype: int64

In [36]:
etats = {
    "SP": "Sao Paulo",
    "RJ": "Rio de Janeiro",
    "MG": "Minas Gerais",
    "RS": "Rio Grande do Sul",
    "PR": "Parana",
    "SC": "Santa Catarina",
    "BA": "Bahia",
    "DF": "Distrito Federal",
    "ES": "Espirito Santo",
    "GO": "Goias",
    "PE": "Pernambuco",
    "CE": "Ceara",
    "PA": "Para",
    "MT": "Mato Grosso",
    "MA": "Maranhao",
    "MS": "Mato Grosso do Sul",
    "PB": "Paraiba",
    "PI": "Piaui",
    "RN": "Rio Grande do Norte",
    "AL": "Alagoas",
    "SE": "Sergipe",
    "TO": "Tocantins",
    "RO": "Rondonia",
    "AM": "Amazonas",
    "AC": "Acre",
    "AP": "Amapa",
    "RR": "Roraima"
}

df["etat_client"] = df["etat_client"].replace(etats)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_state,nombre_articles,...,nombre_vendeurs,first_category,nombre_paiements,montant_paiement,nombre_max_echeances,note_avis,livraison_en_retard,jours_promis,mois_achat,jour_semaine_achat
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,SP,1.0,...,1.0,housewares,3.0,38.71,1.0,4.0,0,15,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,BA,1.0,...,1.0,perfumery,1.0,141.46,1.0,4.0,0,19,7,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,GO,1.0,...,1.0,auto,1.0,179.12,3.0,5.0,0,26,8,2
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,RN,1.0,...,1.0,pet_shop,1.0,72.20,1.0,5.0,0,26,11,5
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,SP,1.0,...,1.0,stationery,1.0,28.62,1.0,5.0,0,12,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96465,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,SP,1.0,...,1.0,health_beauty,1.0,85.08,3.0,5.0,0,18,3,3
96466,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,SP,1.0,...,1.0,baby,1.0,195.00,3.0,4.0,0,23,2,1
96467,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,BA,1.0,...,1.0,home_appliances_2,1.0,271.01,5.0,5.0,0,30,8,6
96468,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,RJ,2.0,...,1.0,computers_accessories,1.0,441.16,4.0,2.0,0,37,1,0


In [2]:
np.random.seed(42)


model = Sequential([
    Dense(16, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1),
])
model.compile(
    input_shape=(4,),
    loss_fn=MSE(),
    optimizer=Adam(),
)


In [ ]:



model.summary()
history = model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(x_val, y_val),
)

loss, accuracy = model.evaluate(x_val, y_val)
predictions = model.predict(x_val[:5])

model.save("regression.weli")
restored_model = load_model("regression.weli", compile=False)
restored_predictions = restored_model.predict(x_val[:5])


